# Korelasyon formülü — Monte Carlo doğrulaması

Bu notebook mevcut projenin kanal üreticisini kullanır. Sabit **W** ve aynı **RIS fazları** için:

1. Merkezlenmiş matris hesabını küçük boyutta dört toplamla karşılaştırır.
2. Mevcut üretim hesabıyla eşdeğerliği kontrol eder.
3. Kompleks kanal ortalaması/varyansını native Monte Carlo ile karşılaştırır.
4. **Güç varyansı–Wick farkını ayrı bir tanı olarak** gösterir.

Matris eşitliği Gaussian kabulü gerektirmez. Mevcut `rho` ikinci dereceden açısal yaklaşım + GH kullanır; native MC orijinal trigonometrik modeli kullanır. Dolayısıyla MC hatası yalnızca toplamların yeniden düzenlenmesini ölçmez.


In [ ]:
from pathlib import Path
import subprocess, sys

BRANCH = "validation/correlation-monte-carlo"
REPO_URL = "https://github.com/OsmanMertYilmaz/Phy_Informed_MIMO_RIS_Env.git"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = Path("/content/Phy_Informed_MIMO_RIS_Env_correlation_mc")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
    if not (REPO / "scripts/validate_correlation_mc.py").is_file():
        raise RuntimeError("Bu dizinde doğrulama scripti yok. REPO için yeni bir dizin seçin.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "matplotlib"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "scripts/validate_correlation_mc.py").is_file()), None)
    if REPO is None:
        raise RuntimeError("Notebook'u bu GitHub deposunun içinde açın veya REPO yolunu ayarlayın.")

print("Repo:", REPO)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


## Deney ayarları

İlk deneme için `NRIS=[8,32]`, `N_MC=8192` seçebilirsin. Büyük deney varsayılanı 32/128/512 RIS ve 65.536 örnektir. Varsayılan bankalar Indoor Office LOS ve NLOS, nT=nR=4 kullanır.

Sonuçlar her örnekleme kontrol noktasında kaydedilir. `RESUME=True` tamamlanan bankaları atlar; yarım banka baştan üretilir. Ayar değiştirdiğinde yeni bir `RUN_NAME` kullan.

`STORAGE='auto'` mevcut Drive bağlantısını kullanır; bağlama başarısızsa yerel kayıtla devam eder. `STORAGE='local'` bağlantıyı hiç denemez. Yerel sonuçlar runtime silinmeden ZIP olarak indirilmelidir. `STORAGE='drive'` Drive olmadan çalışmayı durdurur. Yerelden Drive'a sonradan geçiş önceki sonuçları otomatik taşımaz.


In [ ]:
import torch

NRIS = [32, 128, 512]
N_MC = 65_536
MC_CHUNK = 128
PATTERNS = 4
SEED = 20260910
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FLOAT32 = False
RESUME = True
RUN_NAME = "correlation_mc_64k"
BANK_JSON = None  # Kendi bankalarını içeren JSON yolu; örnek aşağıda.

# "auto": Drive varsa kullan; bağlanamazsa açık uyarıyla yerelde devam et.
# "local": Drive bağlantısını hiç deneme. "drive": Drive olmadan devam etme.
STORAGE = "auto"
DRIVE_MOUNT = Path("/content/drive")


def choose_output_root(in_colab, repo, storage="auto", mount=Path("/content/drive")):
    import tempfile
    if storage not in {"auto", "drive", "local"}:
        raise ValueError("STORAGE auto, drive veya local olmalı")
    local = repo / "outputs"
    if not in_colab or storage == "local":
        return local, False
    from google.colab import drive
    candidates = [mount, Path("/content/google_drive"), Path("/content/correlation_drive")]
    # Önce gerçek ve erişilebilir mevcut bağlantıyı kullan; boş MyDrive klasörü yeterli değil.
    def connected(path):
        try:
            return path.resolve().is_mount() and (path / "MyDrive").is_dir()
        except OSError:
            return False
    # Önceki denemede oluşturulmuş geçici mount noktalarını da yeniden kullan.
    candidates += sorted(Path("/content").glob("correlation_drive_*"))
    existing = next((path for path in candidates if connected(path)), None)
    try:
        if existing is None:
            # Dolu dizinleri silme veya yeniden kullanma; tek temiz hedef seç.
            if mount.is_symlink() or (mount.exists() and
                    (not mount.is_dir() or any(mount.iterdir()))):
                mount = Path(tempfile.mkdtemp(prefix="correlation_drive_", dir="/content"))
            drive.mount(str(mount))
            if not connected(mount):
                raise RuntimeError("Drive bağlandıktan sonra MyDrive erişimi doğrulanamadı")
            existing = mount
        return existing / "MyDrive/Phy_Informed_MIMO_RIS_Env/validation", True
    except (ValueError, OSError, RuntimeError) as exc:
        if storage == "drive":
            raise RuntimeError("Drive bağlanamadı. Yerelde devam için STORAGE='local' seçin.") from exc
        print(f"Drive bağlanamadı: {exc}")
        print("Yerel kayıtla devam ediliyor; sonuçlar henüz Drive'da değil.")
        return local, False


OUTPUT_ROOT, USING_DRIVE = choose_output_root(IN_COLAB, REPO, STORAGE, DRIVE_MOUNT)
OUTPUT = OUTPUT_ROOT / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
if IN_COLAB and not USING_DRIVE:
    print("Yerel dosyalar runtime silinirse kaybolur. Son hücreyle sonuç ZIP'ini indirin.")

print("Device:", DEVICE, "| dtype:", "complex64" if FLOAT32 else "complex128")
print("Output:", OUTPUT)
if DEVICE == "cpu" and max(NRIS) > 32:
    print("CPU için ilk denemede NRIS=[8,32] ve N_MC=8192 daha kısa sürer.")


İstersen aşağıdaki örneği ayrı hücrede çalıştırıp `BANK_JSON` olarak seç. `WIdx` MATLAB'daki gibi 1 tabanlıdır; `nRIS=2*nRIS_x*nRIS_y`.

```python
import json
BANK_JSON = REPO / "outputs/my_correlation_banks.json"
BANK_JSON.parent.mkdir(parents=True, exist_ok=True)
BANK_JSON.write_text(json.dumps([dict(
    scenario_br="Indoor-Office-LOS", scenario_ru="Indoor-Office-NLOS", fc=3.5e9,
    ris=[0,0,3], gnb=[-10,2,3], ue=[8,3,1.5],
    nT1=2, nT2=1, nR1=2, nR2=1,
    nRIS_x=4, nRIS_y=4, WIdx=[1,1,2]
)]))
```


In [ ]:
import os
cmd = [sys.executable, str(REPO / "scripts/validate_correlation_mc.py"),
       "--nris", *map(str, NRIS), "--n-mc", str(N_MC),
       "--mc-chunk", str(MC_CHUNK), "--patterns", str(PATTERNS),
       "--seed", str(SEED), "--device", DEVICE, "--output", str(OUTPUT)]
if FLOAT32:
    cmd.append("--float32")
if RESUME:
    cmd.append("--resume")
if BANK_JSON is not None:
    cmd += ["--bank-json", str(BANK_JSON)]
run_env = os.environ.copy()
run_env["PYTHONPATH"] = str(REPO / "src") + os.pathsep + run_env.get("PYTHONPATH", "")
with subprocess.Popen(cmd, cwd=REPO, env=run_env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
if return_code:
    raise RuntimeError(f"Simülasyon hata kodu {return_code} ile durdu; yukarıdaki çıktıyı inceleyin.")


In [ ]:
import pandas as pd
from IPython.display import display, Image

checks = pd.read_csv(OUTPUT / "algebra_checks.csv")
final = pd.read_csv(OUTPUT / "final_comparison.csv")
display(checks)
summary = final.groupby(["bank", "nris", "scenario_br", "scenario_ru"]).agg(
    channel_var_median_rel_error=("variance_rel_error", "median"),
    channel_var_max_rel_error=("variance_rel_error", "max"),
    mean_error_over_std=("mean_error_over_channel_std", "median"),
    max_abs_residual_se=("centered_residual_se", lambda s: s.abs().max()))
display(summary)
powers = final.drop_duplicates(["bank", "candidate"]).copy()
powers["wick_underestimate_pct"] = 100*(1-powers.wick_power_variance/powers.mc_power_variance)
display(powers[["bank", "nris", "candidate", "analytic_mean_power", "mc_mean_power",
                "mc_power_variance", "wick_power_variance", "wick_underestimate_pct"]])
display(Image(filename=str(OUTPUT / "comparison.png")))


## Sonuçları yorumlama

- `four_sum_*_rel_error`: double precision'da sayısal hassasiyete yakın olmalı. Büyük boyutta güvenlik sınırı nedeniyle hesaplanmaz.
- `production_variance_rel_error`: mevcut üretim hesabıyla fark. Yeni yol negatif varyansı sıfıra kırpmaz.
- `variance_rel_error`: kanal varyansında Monte Carlo farkı. Tablo kesir olarak gösterir; yüzde için 100 ile çarp.
- `centered_residual_se`: analitik ortalama etrafındaki örnek enerjisinin, teorik varyanstan kaç standart hata uzakta olduğu. Kalıcı fark açısal model yaklaşımından gelebilir; tek başına bir doğruluk testi değildir.
- `mc_power_variance` ile `wick_power_variance` **güç varyansıdır**. Kanal varyansı doğru olsa bile aralarında fark olabilir.

[Detaylı açıklama](https://github.com/OsmanMertYilmaz/Phy_Informed_MIMO_RIS_Env/blob/validation/correlation-monte-carlo/docs/CORRELATION_MC.md)


## Sonuçları indir
Özellikle yerel kayıt kullanıyorsan runtime silinmeden bu hücreyi çalıştır. Simülasyon yarıda kalsa bile mevcut ara sonuçları ZIP içine alır.


In [ ]:
import shutil
from IPython.display import FileLink, display

if not OUTPUT.exists() or not any(OUTPUT.iterdir()):
    raise RuntimeError("Henüz indirilecek sonuç yok")
# Arşivi sonuç dizininin dışında oluştur; önceki ZIP arşive tekrar girmez.
archive_root = REPO / "outputs" / "_downloads"
archive_root.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(archive_root / RUN_NAME), "zip", root_dir=OUTPUT)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
else:
    display(FileLink(archive))
